# **Pibic-Pibiti-Huac**: Arquitetura geral para _fine-tuning_ supervisionado de _LLMs_ de código aberto (_open-source_)

Este notebook faz parte de um projeto de pesquisa PIBIC/PIBITI, desenvolvido na Universidade Federal de Campina Grande (UFCG), com foco em conceber e avaliar uma ferramenta, baseda em Inteligência Artificial, para classificação e geração de laudos Radiografia de Tórax produzidos pelo setor de Diagnóstico por Imagem do Hospital Universitário Alcides Carneiro (HUAC).

## Propósito deste _notebook_

O foco deste notebook é construir uma arquitetura geral de _Fine-Tuning_ supervisionado para modelos de linguagem de grande escala (_LLMs_) de código aberto, cuja seleção se deu em uma etapa anterior desta mesma pesquisa.

## Metodologia

Pretende-se utilizar, aqui, técnicas de _Fine-Tuning_ eficiente, como _Low-Rank Adaptation_ (LoRa) e _quantização_.

## Dataset utilizado

Em etapas anteriores desta pesquisa, desenvolvemos _datasets_ que abarcam laudos em imagens advindas de exames de Radiografia de Tórax, devidamente **tratados** e **anonimizados**.

Nesse sentido, em se tratanto de **aprendizagem supervisionada**, utilizaremos um _dataset_ de laudos rotulados segundo três categorias:

 - **OURO**.
 - **PRATA**.
 - **BRONZE**.

Cada categoria reflete, em níveis distintos, o grau de refinamento estrutual e técnico de laudos de Radiografia de Tórax.

O dataset a ser utilizado neste notebook pode ser visualizado [aqui](https://huggingface.co/datasets/guilhermenf/huac_labeled_chest_xray_reports).

## Outros _datasets_

 - [_Dataset_ apenas com laudos de Radiografia de Tórax.](https://huggingface.co/datasets/guilhermenf/huac_chest_xray_reports)
 - [_Dataset_ com laudos e imagens (PA e PERFIL) de Radiografa de Tórax.](https://huggingface.co/datasets/guilhermenf/huac_chest_xray_reports_images)

## Autores

**Guilherme Noronha**, graduando em Ciência da Computação pela Universidade Federal de Campina Grande (UFCG).

 - **Email**: guilherme.noronha.fragoso@ccc.ufcg.edu.br
 - **Github**: [guinoronhaf](https://github.com/guinoronhaf)
 - **HuggingFace**: [guilhermenf](https://huggingface.co/guilhermenf)
 - **Linkedin**: [Guilherme Fragoso](https://www.linkedin.com/in/guilherme-noronha-fragoso/)

**João Ventura**, graduando em Ciência da Computação pela Universidade Federal de Campina Grande (UFCG).

 - **Email**: joao.ventura.crispim.neto@ccc.ufcg.edu.br
 - **Github**: [joaoneto9](https://github.com/joaoneto9)
 - **HuggingFace**: [joaneto9](https://huggingface.co/joaoneto9)
 - **Linkedin**: [João Neto](https://www.linkedin.com/in/jo%C3%A3oneto09/)

## Instalando dependências

In [1]:
# Instala ferramentas necessárias para realização de fine-tuning eficiente supervisionado
# google-cloud-bigquery e sacremoses são necessárias para carregamento de alguns dos modelos selecionados
!pip install -q -U transformers peft accelerate bitsandbytes trl google-cloud-bigquery sacremoses


print("Dependências instaladas!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.3/262.3 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 103.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-

## Login no _HuggingFace_

Antes de tudo, é preciso fazer login, utilizando token de acesso, na plataforma _HuggingFace_, tanto para baixar os modelos quanto para fazer download do dataset de treino.

In [2]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN_GUI") # Obtem HF token do sistema de Secrets do Kaggle

try:
    login(hf_token) # Tenta realizar login no HuggingFace
    print("Login realizado!")
except Exception as exc: # Exceçao em caso de insucesso
    print(f"Algo deu errado: {exc}")

Login realizado!


## Definindo **catálogo** de modelos

Nessa etapa, definiremos, utilizando um dicionário Python, quais serão os modelos efetivamente treinados.

In [3]:
models_dict = {
    "mistral-7b": { # id do modelo
        "path": "mistralai/Mistral-7B-Instruct-v0.3", # url do modelo
        "remote_code": True # acesso a remote_code durante carregamento
    },
    "llama-1b": {
        "path": "meta-llama/Llama-3.2-1B-Instruct",
        "remote_code": True
    },
    "llama-8b": {
        "path": "meta-llama/Llama-3.1-8B-Instruct",
        "remote_code": True
    },
    "llama-3b": {
        "path": "meta-llama/Llama-3.2-3B-Instruct",
        "remote_code": True
    },
    "phi-3-mini-4b": {
        "path": "microsoft/Phi-3-mini-4k-instruct",
        "remote_code": False
    },
    "qwen2.5-3b": {
        "path": "Qwen/Qwen2.5-3B-Instruct",
        "remote_code": True
    },
    "biogpt-large": {
        "path": "microsoft/BioGPT-Large",
        "remote_code": True
    },
    "biomedical-llama-1b": {
        "path": "ContactDoctor/Bio-Medical-Llama-3-2-1B-CoT-012025",
        "remote_code": True
    }
}

print("Catálogo de modelos definido!")

Catálogo de modelos definido!


In [4]:
# Define a lista de modelos possíveis
models_to_chose = list(models_dict.keys())

print(models_to_chose)

['mistral-7b', 'llama-1b', 'llama-8b', 'llama-3b', 'phi-3-mini-4b', 'qwen2.5-3b', 'biogpt-large', 'biomedical-llama-1b']


In [5]:
# Define um modelo para ser alvo do treinamento
model_id = models_to_chose[1]

print(f"Modelo escolhido: {model_id}")

Modelo escolhido: llama-1b


In [6]:
model_path = models_dict[model_id]["path"] # Extrai a url do modelo no HuggingFace
model_remote_code = models_dict[model_id]["remote_code"] # Determina se remote_code será ou não utilizado

## Configurações de treinamento

Agora, com o catálogo de modelos definido, determinaremos as configurações do treinamento visando eficiência: **LoRa** e **quantização**.

### Quantização

Para determinar as configurações de **quantização**, utilizaremos a classe `BitsAndBytesConfig`, disponível via bibliotecas _transformers_ e _bitsandbytes_.

In [7]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Configurações de quantização finalizadas!")

Configurações de quantização finalizadas!


## Carregando modelo e _tokenizer_

Nessa etapa, baixaremos arquivos do repositório do modelo no _HuggingFace_.

### Modelo

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=model_remote_code
)

print("Modelo baixado!")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Modelo baixado!


### _Tokenizer_

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

## Aplicando configurações de _LoRa_

Agora definiremos as configurações de LoRa. Para isso, será preciso definir, para o modelo em questão, quais são os módulos-alvo de treinamento.

### Obtendo **target modules**

In [10]:
import bitsandbytes as bnb
import torch

"""
Obtém os target modules do fine tuning um modelo específico, no formato de lista.

params:
    model: modelo a ter os target modules extraídos
returns:
    lista com os target modules.
"""
def find_all_linear_names(model) -> list():
    cls = bnb.nn.Linear4bit

    lora_module_names = set()

    for name, module in model.named_modules():
        if isinstance(module, cls):
            name_sp = name.split('.')
            lora_module_names.add(name_sp[-1] if len(name_sp) > 1 else name)

    if 'lm_head' in lora_module_names:
        lora_module_names.remove('lm_head')

    return list(lora_module_names)


print("Função para definição dos target modules definida!")

Função para definição dos target modules definida!


In [11]:
model_target_modules = find_all_linear_names(model) # target modules obtidos

In [12]:
print(model_target_modules)

['q_proj', 'o_proj', 'gate_proj', 'v_proj', 'k_proj', 'up_proj', 'down_proj']


### Configurando _LoRa_

In [13]:
import gc
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ----- limpando cache das GPUs -----
gc.collect()
torch.cuda.empty_cache()
# -------------

model = prepare_model_for_kbit_training(model) # Prepara modelo para que o fine tuning quantizado funcione corretamente

# ----- Definição das configurações de LoRa -----
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=model_target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM" # tarefa a ser desempenhada
)

In [14]:
model = get_peft_model(model, peft_config) # modelo preparado para fine tuning eficiente

In [15]:
print("Parâmetros treináveis:")
model.print_trainable_parameters()

Parâmetros treináveis:
trainable params: 11,272,192 || all params: 1,247,086,592 || trainable%: 0.9039


## Preparando _dataset_ de treinamento

Como exposto anteriormente, utilizaremos o _dataset_ de **laudos categorizados** disponível neste [link.](https://huggingface.co/datasets/guilhermenf/huac_labeled_chest_xray_reports)

### Carregando _dataset_

In [16]:
from datasets import load_dataset, Dataset

# Path do dataset
dataset_path = "guilhermenf/huac_labeled_chest_xray_reports"

# Carrega o dataset completo
ds = load_dataset(dataset_path)

ds_train = ds["train"] # Dataset para treino
ds_test = ds["test"] # Dataset para teste

README.md:   0%|          | 0.00/505 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/23.0k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/13.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/75 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/26 [00:00<?, ? examples/s]

### Criando amostras de dados a serem injetadas no treinamento

Nessa etapa, criaremos uma função, chamada `crate_sample()`, que será responsável por receber um conjunto de elementos do dataset e convertê-lo em uma estrutura adequada para injeção no modelo durante o treinamento.

In [17]:
system_prompt = """
Você é um Radiologista Analista Sênior especializado em auditoria estrutural de laudos radiológicos.

Você deve avaliar a QUALIDADE ESTRUTURAL de um laudo de Radiografia de Tórax.

Avalie:
- cobertura estrutural do laudo;
- formalismo técnico;
- especificidade anatômica;
- completude de descrição.

Estrura da avaliação:
- Você deve atribuir uma classificação (label) ao laudo -> OURO | PRATA | BRONZE.
- Você deve justificar sua classificação a partir de elementos do laudo -> feedback.

Não avalie diagnóstico médico.
"""

In [18]:
import json

"""
Recebe um conjunto de dados do dataset, e tokeniza os dados de cada um dos elementos utilizando o tokenizer do modelo em questão.

params:
    sample: conjunto de dados.
returns:
    dicionáiro contendo dados devidamente tokenizados.
"""
def create_sample(sample) -> dict():
    global tokenizer
    # ---- Obtém colunas do dataset para treinamento supervisionado ----
    report_content = sample["report"]
    report_label = sample["label"]
    report_feedback = sample["feedback"]
    # ----
    texts = []

    for report, label, feedback in zip(report_content, report_label, report_feedback): # Associa cada informação a um laudo
        assistant_content = json.dumps({ # Gera uma string, representando um JSON, que contém as informações de classificação (label) e feedback do laudo.
            "label": label,
            "feedback": feedback
        }, ensure_ascii=False)
        
        chat = [
            {"role": "system", "content": system_prompt}, # system_prompt (Instruções)
            {"role": "user", "content": report}, # user_prompt (Laudo)
            {"role": "assistant", "content": assistant_content} # assistant (Label e feedback)
        ]

        tokenized_data = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False) + tokenizer.eos_token # Aplica chat_template ao chat
        texts.append(tokenized_data) # Adiciona chat tokenizado à lista texts

    return { "text": texts }


print("Função para tokenização do dataset definida!")

Função para tokenização do dataset definida!


In [19]:
dataset_train = ds_train.map(create_sample, batched=True, remove_columns=ds_train.column_names) # Mapeia o dataset, aplicando a função definida anteriormente

Map:   0%|          | 0/75 [00:00<?, ? examples/s]

In [20]:
dataset_train = dataset_train.select_columns(["text"]) # Filtra apenas a coluna de interesse ("text")

In [21]:
print(dataset_train[0]) # Exemplo do primeiro elemento do dataset tokenizado segundo o chat_template

{'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 25 May 2026\n\nVocê é um Radiologista Analista Sênior especializado em auditoria estrutural de laudos radiológicos.\n\nVocê deve avaliar a QUALIDADE ESTRUTURAL de um laudo de Radiografia de Tórax.\n\nAvalie:\n- cobertura estrutural do laudo;\n- formalismo técnico;\n- especificidade anatômica;\n- completude de descrição.\n\nEstrura da avaliação:\n- Você deve atribuir uma classificação (label) ao laudo -> OURO | PRATA | BRONZE.\n- Você deve justificar sua classificação a partir de elementos do laudo -> feedback.\n\nNão avalie diagnóstico médico.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n- Transparência pulmonar normal.\n- Mediastino sem alterações.\n- Imagem cardíaca de configuração e dimensões normais.\n- Seios costo-frênicos livres.\n- Espondilose dorsal. incipiente.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{"label": "PRATA", "feedback"

## Treinamento do modelo

Agora, iremos de fato realizar o treinamento do modelo em questão utilizando o _dataset_ de treino recém-gerado. Nesta etapa, utilizaremos a técnica de **Fine-Tuning Supervisionado** (_Supervisioned Fine-Tuning_), em que injetamos no modelo os dados **rotulados**. Nesse caso, a rotulação se dá a partir da **Label**, isto é, a classificação estrutural de cada laudo em **OURO**, **PRATA** ou **BRONZE**.

### Definição dos parâmetros de treinamento

In [22]:
output_dir = f"/kaggle/working/{model_id}_fine_tuning/"

In [23]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    num_train_epochs=3.0,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    gradient_accumulation_steps=16,
    logging_strategy="steps",
    logging_steps=1,
    gradient_checkpointing=True,
)

print("Configurações de treinamento definidas!")

Configurações de treinamento definidas!


### Realizando treinamento

In [24]:
import torch
from trl import SFTTrainer

torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train
)

print("Configurações finais de treinamento feitas!")

trainer.train()

Adding EOS to train dataset:   0%|          | 0/75 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/75 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Configurações finais de treinamento feitas!


Step,Training Loss
1,6.338247
2,5.352642
3,4.699049
4,4.246381
5,3.740849
6,3.354465
7,3.202928
8,2.921751
9,2.617828


TrainOutput(global_step=9, training_loss=4.052682240804036, metrics={'train_runtime': 445.5929, 'train_samples_per_second': 0.505, 'train_steps_per_second': 0.02, 'total_flos': 573917840068608.0, 'train_loss': 4.052682240804036})

## Realizando inferências

Aqui, realizaremos algumas inferências pontuais para testar o desempenho do modelo, em caráter preliminar.

In [25]:
# Obtém elemento inicial do dataset de treino
test_ds_element = ds_test[0]

print(test_ds_element)

{'id': '42', 'requisition': '2601140851491107', 'report': '- Transparência pulmonar preservada.\n- Seios costofrênicos livres.\n- Mediastino sem alterações.\n- Índice cardiotorácico dentro da normalidade.\n- Retificação da cifose dorsal fisiológica.\n- Espondilose dorsal.', 'label': 'PRATA', 'feedback': 'A avaliação da coluna dorsal é parcial e não fornece detalhes completos sobre todo o arcabouço ósseo, resultando em nota 0.5 para o critério 1; As avaliações dos pulmões e pleura são adequadas (critério 2); O ICT e mediastino foram avaliados corretamente (critério 3); A descrição das alterações existentes é objetiva e localizada, resultando em nota 1.0 para o critério 4.'}


In [26]:
test_chat = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_ds_element["report"]}
]

print("Chat de teste definido!")

Chat de teste definido!


In [27]:
text = tokenizer.apply_chat_template(
    test_chat,
    tokenize=False,
    add_generation_prompt=False
)

print("Aplicação do chat_template concluída!")

Aplicação do chat_template concluída!


In [28]:
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

print("Tokenização aplicada!")

Tokenização aplicada!


In [29]:
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=False
)

print("Geração da resposta concluída!")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in LlamaDecoderLayer. Setting `past_key_values=None`.


Geração da resposta concluída!


In [30]:
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=False
)

print(response)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 25 May 2026

Você é um Radiologista Analista Sênior especializado em auditoria estrutural de laudos radiológicos.

Você deve avaliar a QUALIDADE ESTRUTURAL de um laudo de Radiografia de Tórax.

Avalie:
- cobertura estrutural do laudo;
- formalismo técnico;
- especificidade anatômica;
- completude de descrição.

Estrura da avaliação:
- Você deve atribuir uma classificação (label) ao laudo -> OURO | PRATA | BRONZE.
- Você deve justificar sua classificação a partir de elementos do laudo -> feedback.

Não avalie diagnóstico médico.<|eot_id|><|start_header_id|>user<|end_header_id|>

- Transparência pulmonar preservada.
- Seios costofrênicos livres.
- Mediastino sem alterações.
- Índice cardiotorácico dentro da normalidade.
- Retificação da cifose dorsal fisiológica.
- Espondilose dorsal.<|eot_id|><|start_header_id|>QuestionQuestionQuestionQuestionQuestionQuestionQu